# The covariance of DESI $P_\ell(k)$, built term by term with thecov

For one tracer and cap, this notebook builds each piece of the covariance with explicit thecov calls, and after each
piece compares with the mocks:

1. **the mocks**: what we want to reproduce;
2. **the window** $m(\mathbf x)$ from the randoms: each random's own weight vs the local mean weight;
3. **the Gaussian term**: local window $m^2$, then the $\xi$-kernel (pair-averaged) window;
4. **linear theory** for the non-Gaussian terms ($P_{\rm lin}$, $b_1$, $f$, damping, amplitude);
5. **super-sample covariance**: long-mode variances, responses, the local average; a direct test with each mock's normalisation;
6. **discreteness** four-point terms;
7. *(optional)* the connected trispectrum $T_0$;
8. **comparisons**: each term's size, diagonals, $\chi^2$, the modes of the covariance, parameter errors, fitted amplitudes.

Only the file reading is hidden (`paperdata.load_mock_spectra`, `paperdata.load_catalogs`). Everything is
computed here. The expensive part of thecov, the pair counts of the window (`cov.compute_windows`: minutes for the
local window, about an hour for the $\xi$ kernel), is saved in `CACHE` on the first run and read back afterwards.

Run on NERSC Jupyter, DESI kernel, preferably an exclusive CPU node. Change `TRACER` / `CAP` below and run all.

## 0. Settings

In [ ]:
TRACER, CAP = 'LRG1', 'NGC'           # 'LRG1' or 'QSO'; 'NGC' or 'SGC'
DROP_DEFECTIVE = True                 # leave out the holi mocks with a broken n(z) (needs every mock's catalogue once)
RUN_T0 = False                        # the connected trispectrum (section 7): ~10 min with 64 processes

import os, sys
HOME = os.path.expanduser('~')
sys.path.insert(0, HOME + '/thecov/paper')
sys.path.insert(0, HOME + '/thecov')
import numpy as np
import matplotlib.pyplot as plt
import paperdata
from paperdata import load_mock_spectra, load_catalogs
from desi_validation import desi_compare as dc      # sky_to_cartesian, random_density, local_mean_weight, ...
import thecov
from thecov import (Tracer, PowerSpectrumModel, GaussianCovariance, WindowSmoothing, SuperSampleCovariance,
                    DiscretenessCovariance)
from thecov.covariance_tools import window_convolve
%matplotlib inline

# this notebook's pair counts: computed on the first run (the expensive part), read back on later runs.
# Delete the directory (or change the settings below, which changes nothing in the cache: delete it) to recompute.
CACHE = os.environ.get('SCRATCH', HOME) + '/thecov_walkthrough'
SURFACE_DENSITY = 2500.0                                # randoms per deg^2 per random file (DESI)
N_RANDOMS_LOCAL = 2_000_000                             # randoms of the local-window and SSC tracers
N_RANDOMS_KERNEL = 4_000_000                            # randoms of the xi-kernel tracer
KERNEL_RANDOM_FILES = 10                                # random files sampling m densely for the xi-kernel smoothing
NEAR_PAIRS_LOCAL, NEAR_PAIRS_KERNEL = 2e9, 1e10         # pair-count sampling at s < 80 Mpc/h (accuracy vs time)
paperdata.CACHE_DIR = os.environ.get('SCRATCH', HOME) + '/paper_nersc_cache'
RESULTS = HOME + '/thecov/paper/walkthrough'            # this notebook's matrices, for the paper figures
os.makedirs(RESULTS, exist_ok=True)

In [ ]:
# ---- look ----
MOCKS, OWN, LOCAL, GAUSS, FINAL = '#0b0b0b', '#898781', '#c3c2b7', '#2a78d6', '#eb6834'
SSC, DISC, T0, LIGHT = '#1baf7a', '#e87ba4', '#6250d6', '#e1e0d9'
ELL_LINE = {0: '-', 2: '--', 4: ':'}
K_LABEL = r'$k\ [h\,\mathrm{Mpc}^{-1}]$'
WIDTH = 6.3
plt.rcParams.update({
    'font.family': 'serif', 'mathtext.fontset': 'cm', 'font.size': 9, 'axes.titlesize': 9, 'legend.fontsize': 8,
    'axes.titlelocation': 'left', 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
    'grid.color': LIGHT, 'grid.linewidth': 0.6, 'legend.frameon': False, 'lines.linewidth': 1.6, 'figure.dpi': 130,
    'figure.constrained_layout.use': True})

## 1. The mocks

Each mock gives a data vector $(P_0, P_2, P_4)$ at $n_k$ bins each. Their sample covariance $S$ is what every model
below is compared with. Its noise: each element scatters by $\sigma_W=\sqrt{(C_{ii}C_{jj}+C_{ij}^2)/(N-1)}$.

In [ ]:
spec = load_mock_spectra('holi', TRACER, CAP, drop_defective=DROP_DEFECTIVE)
k, n_k, N = spec.k, len(spec.k), len(spec.mocks)
P0, P2, P4 = (slice(0, n_k), slice(n_k, 2 * n_k), slice(2 * n_k, 3 * n_k))     # where each multipole sits
mean = spec.mocks.mean(0)
S = np.cov(spec.mocks.T)                       # sample covariance of the mocks
norm = spec.norm.mean()                        # the estimator's normalisation (mean over mocks)
print(f'{spec.name}: {N} mocks, {n_k} k bins per multipole ({k[0]:.4f} ... {k[-1]:.4f} h/Mpc), norm {norm:.4g}')

fig, (left, right) = plt.subplots(1, 2, figsize=(WIDTH, 2.4))
for ell, part in zip((0, 2, 4), (P0, P2, P4)):
    left.plot(k, k * mean[part], color=MOCKS, ls=ELL_LINE[ell], label=rf'$P_{ell}$')
    left.fill_between(k, k * (mean[part] - np.sqrt(np.diag(S)[part])), k * (mean[part] + np.sqrt(np.diag(S)[part])),
                      color=LIGHT, lw=0)
left.set(xlabel=K_LABEL, ylabel=r'$k P_\ell$', title='mean of the mocks (band: one mock\'s scatter)')
left.legend()
d = np.sqrt(np.diag(S))
im = right.imshow(S / np.outer(d, d), cmap='RdBu_r', vmin=-0.4, vmax=0.4, origin='lower')
right.set(title='correlation matrix of the mocks', xticks=[n_k / 2, 1.5 * n_k, 2.5 * n_k], yticks=[n_k / 2, 1.5 * n_k, 2.5 * n_k])
right.set_xticklabels([r'$P_0$', r'$P_2$', r'$P_4$']); right.set_yticklabels([r'$P_0$', r'$P_2$', r'$P_4$'])
right.grid(False)
fig.colorbar(im, ax=right, fraction=0.046)

## 2. The window $m(\mathbf x)$ from the randoms

The estimator weights each galaxy and random by $w = $ `WEIGHT` $\times$ `WEIGHT_FKP`. The clustering window is
$m(\mathbf x)=\bar n(\mathbf x)\,\langle w\rangle(\mathbf x)$, the mean weighted density, sampled at the randoms.

Both constructions below use the same density, $\alpha\,\rho_r(z)$ ($\rho_r$: the randoms per unit volume), and differ
only in the weight:

* **Own weight (no fix):** $m_r=\alpha\,\rho_r(z)\,w_r$, each random with its own weight. The window then contains
  $m^2\propto w_r^2$, i.e. $\langle w^2\rangle$ where the clustering of two *different* galaxies needs $\langle w\rangle^2$:
  a self-pair that inflates the Gaussian term by up to $\langle w^2\rangle/\langle w\rangle^2$.
* **Local mean weight:** $m_r=\alpha\,\rho_r(z)\,\langle w\rangle_{32}$, with $\langle w\rangle_{32}$ the mean weight of the
  32 nearest *other* randoms.

(An earlier version took $m=$ `NX` $\times w_r$ for the own-weight case. `NX` is already the completeness-weighted density,
so that multiplied in `WEIGHT` a second time, and differed from the mean-weight window by ~10% in amplitude and across
completeness regions, not only in the weights.)

**About thecov's alpha warning** (ratio ~1.2 for LRG). thecov checks $m$ against a 64-nearest-neighbour density of the
randoms, over spheres of ~20 $h^{-1}$Mpc (~0.9° at $z=0.5$). Those spheres contain veto holes, while $m$ is the density
between the holes, so the ratio is ~1/(unvetoed fraction at that scale): 1/0.816 = 1.22 for LRG1 NGC, from the hole
statistics of the randoms. That is the footprint, not an error in $m$.

In [ ]:
cat = load_catalogs(TRACER, CAP, n_random_files=1)          # data and randoms of mock 173, z-cut
w_data = cat.data['WEIGHT'] * cat.data['WEIGHT_FKP']
w_rand = cat.randoms['WEIGHT'] * cat.randoms['WEIGHT_FKP']
alpha_all = w_data.sum() / w_rand.sum()
positions = dc.sky_to_cartesian(cat.randoms['RA'], cat.randoms['DEC'], cat.randoms['Z'])
rho_r, _ = dc.random_density(cat, SURFACE_DENSITY)          # randoms per (Mpc/h)^3 at each random
print(f'{len(w_data)} galaxies, {len(w_rand)} randoms, alpha = {alpha_all:.4f}; '
      f'<w^2>/<w>^2 of the randoms = {np.mean(w_rand**2) / np.mean(w_rand)**2:.4f}')

def make_tracer(name, weights, n_randoms, shot_noise='spectra'):
    # a thecov Tracer = randoms (positions, weights) + the window m at each random + alpha + shot-noise scale
    keep = np.random.default_rng(0).random(len(w_rand)) < min(1.0, n_randoms / len(w_rand))
    w = w_rand[keep]
    alpha = w_data.sum() / w.sum()
    randoms = {'POSITION': positions[keep], 'WEIGHT': w}
    if weights == 'own':
        randoms['NW'] = alpha_all * rho_r[keep] * w                             # each random's own weight
    else:
        randoms['NW'] = alpha_all * rho_r[keep] * dc.local_mean_weight(positions[keep], w, k=32)
    # shot noise: thecov's S window integrates to (1 + alpha) alpha sum_r w^2; rescale it to the measured one
    target = spec.num_shotnoise.mean() if shot_noise == 'spectra' else np.sum(w_data**2) + alpha_all**2 * np.sum(w_rand**2)
    return Tracer(name, randoms, alpha, shotnoise_scale=target / ((1 + alpha) * alpha * np.sum(w**2)))

tracer_own = make_tracer(f'{TRACER}_{CAP}_own', 'own', N_RANDOMS_LOCAL)
tracer_mean = make_tracer(f'{TRACER}_{CAP}', 'mean', N_RANDOMS_LOCAL)

ratio = tracer_own.mw / tracer_mean.mw                      # m_own / m_mean at the same randoms
fig, ax = plt.subplots(figsize=(WIDTH / 2, 2.3))
ax.hist(ratio, bins=np.linspace(0.5, 1.8, 80), color=GAUSS, histtype='stepfilled', alpha=0.6)
ax.set(xlabel=r'$m_{\rm own} / m_{\rm mean}$ at each random', ylabel='randoms', yticks=[],
       title=rf'own / local mean weight: $\langle r^2\rangle/\langle r\rangle^2$ = {np.mean(ratio**2) / np.mean(ratio)**2:.3f}')

## 3. The Gaussian term

`GaussianCovariance` computes $C^{\rm G}$ for a windowed estimator in separation space: pair counts of the window
(*tripolar window functions*, the expensive geometry part), combined with the power spectrum. Three inputs fix it:
the window (the tracer), the estimator's normalisation (`set_normalization`), and the power spectrum. Here the power
spectrum is the mean of the mocks; since it is window-convolved, `masked=True` tells thecov to divide by the window's
dilution $I_k/\mathcal N$.

In [ ]:
def model_from_mocks(name):
    # P_ell(k) = mean of the mocks, extended flat to k = 0 and to the last bin edge
    kk = np.r_[0.0, k, spec.k_edges[-1]]
    model = PowerSpectrumModel()
    model.add((name, name), {ell: (kk, np.r_[mean[part][0], mean[part], mean[part][-1]])
                             for ell, part in zip((0, 2, 4), (P0, P2, P4))})
    return model

def gaussian_covariance(tracer, smoothing=None, target_near_pairs=NEAR_PAIRS_LOCAL):
    # thecov's Gaussian covariance for this tracer's window, on the mocks' k bins
    windows_file = f'{CACHE}/{TRACER}/windows_{tracer.name}.npz'
    cached = os.path.exists(windows_file)
    n_near = 200_000 if cached else dc.suggest_n_near(tracer, target_pairs=target_near_pairs)[0]
    cov = GaussianCovariance([tracer], spec.k_edges, ells=(0, 2, 4), L_max=4, ds=2.0, ds_pair=10.0, s_split=80.0,
                             n_sub=20000, n_near=n_near, shot_noise=True, smoothing=smoothing)
    if cached:
        cov.load_windows(windows_file)                           # pair counts of this notebook's first run
        print('pair counts read from', windows_file)
    else:
        print('computing the pair counts (minutes to an hour) ...')
        cov.compute_windows([(tracer.name, tracer.name)], verbose=True)
        os.makedirs(os.path.dirname(windows_file), exist_ok=True)
        cov.save_windows(windows_file)
    cov.set_normalization(tracer.name, tracer.name, norm)
    cov.set_model(model_from_mocks(tracer.name), masked=True)
    C, _ = cov.covariance([(tracer.name, tracer.name)])
    return C, cov

C_own, cov_own = gaussian_covariance(tracer_own)
C_mean, cov_mean = gaussian_covariance(tracer_mean)

### 3b. The $\xi$-kernel window

The local window takes $m(\mathbf x)\,m(\mathbf x+\mathbf r)\simeq m^2(\mathbf x)$ for every correlated partner. Keeping the
partner gives $W_k=m\,(K_k*m)$ with $K_k(r)=\xi_0(r)\,j_0(kr)/P_0(k)$, normalised to $\int K_k=1$. Where $m$ is smooth
nothing changes; near veto holes and edges, the partners missing in the holes lower $W_k$. The variation of that
fraction across the footprint raises the variance. First the kernels, from the mocks' mean monopole:

In [ ]:
from thecov.smoothing import extend_power, xi0_from_power, bin_kernels
r = np.arange(0.125, 200, 0.25)
kk, pk = extend_power(k, mean[P0], damping=1.0)          # P_0 extended to k = 5 h/Mpc, damped beyond the data
xi0 = xi0_from_power(kk, pk, r)
kernels = bin_kernels(spec.k_edges, r, xi0)               # one kernel per k bin, int 4 pi r^2 K dr = 1

fig, ax = plt.subplots(figsize=(WIDTH / 2, 2.4))
for i, color in zip((0, n_k // 4, n_k // 2, n_k - 1), ('#0d366b', '#184f95', '#3987e5', '#9ec5f4')):
    ax.plot(r, 4 * np.pi * r**2 * kernels[i], color=color, label=f'k = {k[i]:.3f}')
ax.axhline(0, color=MOCKS, lw=0.6)
ax.set(xlim=(0, 120), xlabel=r'$r\ [h^{-1}\mathrm{Mpc}]$', ylabel=r'$4\pi r^2 K_k(r)$', title='the kernels')
ax.legend()

In [ ]:
# the smoothing: m sampled densely (many random files) and smoothed with the kernels; then the pair counts of the
# smoothed windows (expanded on a small basis). Saved with the pair counts and read back on later runs.
kernel_name = f'{TRACER}_{CAP}_kernel'
tracer_kernel = make_tracer(kernel_name, 'mean', N_RANDOMS_KERNEL)
smoothing = WindowSmoothing(cell=4.0, r_split=8.0, r_max=200.0, tol=2e-3, max_kernels=48, damping=1.0)
if not os.path.exists(f'{CACHE}/{TRACER}/windows_{kernel_name}.npz.smoothing.npz'):
    ref = load_catalogs(TRACER, CAP, n_random_files=KERNEL_RANDOM_FILES)
    w_ref = ref.randoms['WEIGHT'] * ref.randoms['WEIGHT_FKP']
    smoothing.add_density(kernel_name, dc.sky_to_cartesian(ref.randoms['RA'], ref.randoms['DEC'], ref.randoms['Z']),
                          w_ref, w_data.sum() / w_ref.sum())
    smoothing.set_power(kernel_name, kernel_name, k, mean[P0])
    del ref
C_kernel, cov_kernel = gaussian_covariance(tracer_kernel, smoothing=smoothing, target_near_pairs=NEAR_PAIRS_KERNEL)

I_k = cov_kernel.I_k(kernel_name, kernel_name)            # int m (K_k * m), per k bin
I_local = cov_kernel.I_local(kernel_name, kernel_name)    # int m^2
fig, ax = plt.subplots(figsize=(WIDTH / 2, 2.3))
ax.plot(k, I_k / I_local, color=GAUSS)
ax.set(xlabel=K_LABEL, ylabel=r'$\int m(K_k*m)\,/\int m^2$', title='fraction of partners inside the survey')

In [ ]:
# comparison with the matrices of the earlier pipeline runs, when they exist (independent pair counts: expect
# differences at the level of their Monte-Carlo noise, ~1e-3, plus the runs' model and norm, which used all mocks)
for name, C, window in (('local mean weight', C_mean, 'random-density'), ('xi kernel', C_kernel, 'kernel')):
    stored = paperdata.stored_gaussian('holi', TRACER, CAP, window)
    if stored is not None:
        print(f'{name:18s}: max |diag ratio - 1| vs the stored run = {np.max(np.abs(np.diag(C) / np.diag(stored) - 1)):.2e}')

### 3c. What the window choices do, against the mocks

Left: each model's variance relative to the no-fix model. Right: variance of the mocks over each model's, averaged
over 4 neighbouring bins (1 = right). The non-Gaussian terms are still missing here, so at high $k$ all three fall short;
compare them at low $k$, or after section 6 (where the same SSC + discreteness is added to each).

In [ ]:
from paperstats import variance_ratio_error
GROUP = 4
def binned(ratio, C, part):
    # mean of a per-bin ratio over groups of GROUP bins of one multipole, with the expected scatter for N mocks
    first = part.start
    kc, y, e = [], [], []
    for g in range(n_k // GROUP):
        bins = first + np.arange(g * GROUP, (g + 1) * GROUP)
        kc.append(k[bins - first].mean()); y.append(ratio[bins].mean()); e.append(variance_ratio_error(C, N, bins))
    return np.array(kc), np.array(y), np.array(e)

def plot_vs_mocks(models, title):
    fig, axes = plt.subplots(1, 3, figsize=(WIDTH, 2.4), sharey=True)
    for ax, ell, part in zip(axes, (0, 2, 4), (P0, P2, P4)):
        for j, (label, C, color) in enumerate(models):
            kc, y, e = binned(np.diag(S) / np.diag(C), C, part)
            ax.errorbar(kc + (j - (len(models) - 1) / 2) * 0.0015, y, e, fmt='o', ms=3, lw=1, color=color, label=label)
        ax.axhline(1, color=MOCKS, lw=0.7)
        ax.set(xlabel=K_LABEL, title=rf'$P_{ell}$')
    axes[0].set_ylabel('mock / model variance')
    fig.legend(*axes[0].get_legend_handles_labels(), loc='lower center', ncol=len(models), bbox_to_anchor=(0.5, 1.0),
               title=title)

fig, ax = plt.subplots(figsize=(WIDTH / 2, 2.4))
for ell, part in zip((0, 2, 4), (P0, P2, P4)):
    ax.plot(k, np.diag(C_mean)[part] / np.diag(C_own)[part], color=LOCAL, ls=ELL_LINE[ell])
    ax.plot(k, np.diag(C_kernel)[part] / np.diag(C_own)[part], color=GAUSS, ls=ELL_LINE[ell])
ax.plot([], [], color=LOCAL, label='local mean weight'); ax.plot([], [], color=GAUSS, label=r'+ $\xi$ kernel')
ax.axhline(1, color=MOCKS, lw=0.7)
ax.set(xlabel=K_LABEL, ylabel='variance / own-weight variance', title='model vs model (solid/dashed/dotted: P0/P2/P4)')
ax.legend()

plot_vs_mocks([('own weight', C_own, OWN), ('local mean weight', C_mean, LOCAL), (r'$\xi$ kernel', C_kernel, GAUSS)],
              'Gaussian term only')

### 3d. The arbiter: Gaussian random fields on the real footprint

The mocks cannot settle the window at the few-% level: their extra variance also contains SSC, discreteness and the
radial integral constraint. Gaussian random fields can. `desi_validation/run_gaussian_footprint.py` draws fields with
the mocks' $P_0$ (isotropic: $P_2=P_4=0$), multiplies them by a high-resolution window (healpix nside 512 $\times$ the
weighted $n(z)$, on a mesh) with the local mean weight, adds white noise with the shot-noise density, and estimates
$P_0$ as the pipeline does. The variance over realisations is the exact Gaussian covariance for that window: no
local approximation, no kernel, no non-Gaussian terms. Below, each thecov window with the same isotropic model, against it.

Limits: holes smaller than the mesh cell (6 $h^{-1}$Mpc by default) are averaged over the cell (rerun with `--cell 4` to
check convergence); only $P_0$; the fields have no integral constraint (bins $k\lesssim0.03$ are not comparable).

In [ ]:
import glob
GRF_DIR = paperdata.THECOV_CACHE['holi'] + f'/{TRACER}'      # where run_gaussian_footprint.py writes (--out-dir)
grf_files = sorted(glob.glob(f'{GRF_DIR}/gaussian_footprint_{TRACER}_{CAP}_c*_task*.npz'))
print(len(grf_files), 'GRF files:', [os.path.basename(f) for f in grf_files])
if grf_files:
    grf = [np.load(f) for f in grf_files]
    P0_grf = np.concatenate([g['P0_maps_x1'] for g in grf])     # (realisations, n_k)
    assert np.allclose(grf[0]['k_x1'], k), 'GRF k bins differ from the spectra'
    n_grf = len(P0_grf)
    var_grf = P0_grf.var(0, ddof=1)
    print(f'{n_grf} realisations; <P0>_GRF / <P0>_mocks at k > 0.03: '
          f'{np.mean(P0_grf.mean(0)[k > 0.03] / mean[P0][k > 0.03]):.4f}')

    def isotropic_model(name):
        # the mocks' mean P0 only (P2 = P4 = 0), as the random fields
        kk = np.r_[0.0, k, spec.k_edges[-1]]
        model = PowerSpectrumModel()
        model.add((name, name), {0: (kk, np.r_[mean[P0][0], mean[P0], mean[P0][-1]]), 2: (kk, 0 * kk), 4: (kk, 0 * kk)})
        return model

    var_iso = {}
    for label, cov, tracer in (('own weight', cov_own, tracer_own), ('local mean weight', cov_mean, tracer_mean),
                               (r'$\xi$ kernel', cov_kernel, tracer_kernel)):
        cov.set_model(isotropic_model(tracer.name), masked=True)
        C_iso, _ = cov.covariance([(tracer.name, tracer.name)])
        var_iso[label] = np.diag(C_iso)[P0]
        cov.set_model(model_from_mocks(tracer.name), masked=True)      # back to the full model

    fig, (left, right) = plt.subplots(1, 2, figsize=(WIDTH, 2.5))
    groups = [np.arange(g * GROUP, (g + 1) * GROUP) for g in range(n_k // GROUP)]
    kc = np.array([k[b].mean() for b in groups])
    err = np.sqrt(2 / (n_grf - 1) / GROUP)                           # approximate: neighbouring bins are correlated
    for (label, v), color, shift in zip(var_iso.items(), (OWN, LOCAL, GAUSS), (-0.0015, 0, 0.0015)):
        ratio = np.array([np.mean(var_grf[b] / v[b]) for b in groups])
        left.errorbar(kc + shift, ratio, err, fmt='o', ms=3, lw=1, color=color, label=label)
        print(f'{label:20s} Var_GRF / Var_thecov: ' + ', '.join(f'{lo}-{hi}: {np.mean(var_grf[(k >= lo) & (k < hi)] / v[(k >= lo) & (k < hi)]):.3f}'
                                                       for lo, hi in ((0.03, 0.1), (0.1, 0.2), (0.2, 0.3)) if np.any((k >= lo) & (k < hi))))
    left.axhline(1, color=MOCKS, lw=0.7)
    left.set(xlabel=K_LABEL, ylabel='Var GRF / Var thecov', title=r'$P_0$ variance: exact vs thecov')
    left.legend(fontsize=7)
    # neighbouring-bin correlation: how the window mixes adjacent k bins
    corr_grf = np.corrcoef(P0_grf.T)
    for (label, cov, tracer), color in zip((('local mean weight', cov_mean, tracer_mean), (r'$\xi$ kernel', cov_kernel, tracer_kernel)),
                                           (LOCAL, GAUSS)):
        cov.set_model(isotropic_model(tracer.name), masked=True)
        C_iso, _ = cov.covariance([(tracer.name, tracer.name)])
        cov.set_model(model_from_mocks(tracer.name), masked=True)
        c_iso = C_iso[P0, P0] / np.sqrt(np.outer(np.diag(C_iso)[P0], np.diag(C_iso)[P0]))
        right.plot(k[:-1], np.diag(c_iso, 1), color=color, label=label)
    right.plot(k[:-1], np.diag(corr_grf, 1), 'o', ms=2.5, color=MOCKS, label='GRF')
    right.set(xlabel=K_LABEL, ylabel=r'corr($P_0(k_i), P_0(k_{i+1})$)', title='adjacent-bin correlation')
    right.legend(fontsize=7)
else:
    print('no random-field realisations yet: run desi_validation/run_gaussian_footprint.py (see paper/README.md)')

## 4. Linear theory for the non-Gaussian terms

The non-Gaussian terms are built from a linear galaxy power spectrum: $P_{\rm lin}$ of the DESI fiducial cosmology at
$z_{\rm eff}$, growth rate $f$; $b_1$ from the mocks' $P_2/P_0$ at $k<0.06$ (Kaiser); $b_2$ from the Lazeyras et al.
$b_2(b_1)$; $b_{s^2}=-\frac47(b_1-1)$; BAO damping ($\Sigma=6\,h^{-1}$Mpc) and fingers of God ($\sigma_v$, fitted to $P_2/P_0$);
and an amplitude $A$ that matches the mocks' $P_0$ at $0.02<k<0.08$.

In [ ]:
from cosmoprimo.fiducial import DESI
from scipy.interpolate import CubicSpline
from thecov.power import no_wiggle, ir_damped, Dressed
from desi_validation.ssc_check import fit_b1, fit_sigma_v, fit_amplitude, Z_EFF

z_eff = Z_EFF[TRACER]
cosmo = DESI()
k_lin = np.logspace(-4.5, 1.0, 3000)
P_lin = np.asarray(cosmo.get_fourier().pk_interpolator().to_1d(z=z_eff)(k_lin))
f = float(cosmo.growth_rate(z_eff))
spline = CubicSpline(np.log(k_lin), P_lin)
P_lin_at = lambda q: spline(np.log(q))

b1, _ = fit_b1(k, mean[P0], mean[P2], P_lin_at, f)
b2 = 0.412 - 2.143 * b1 + 0.929 * b1**2 + 0.008 * b1**3
bs2 = -4 / 7 * (b1 - 1)
sigma_v = fit_sigma_v(k, mean[P0], mean[P2], b1, f)
P_wiggle_damped = ir_damped(k_lin, P_lin, no_wiggle(k_lin, P_lin, cosmo.h, cosmo.Omega0_m * cosmo.h**2,
                                                    cosmo.Omega0_b / cosmo.Omega0_m, n_s=cosmo.n_s), 6.0)
dilution = I_k / norm                                       # the window's dilution of the measured spectrum
A = fit_amplitude(k, mean[P0], dilution, b1, f, sigma_v, P_lin_at)
P_model = A * P_wiggle_damped                               # what every non-Gaussian term is built from
dressed = Dressed(k_lin, P_model, sigma_v)
print(f'z_eff {z_eff}, f {f:.3f}, b1 {b1:.3f}, b2 {b2:.3f}, bs2 {bs2:.3f}, sigma_v {sigma_v:.2f} Mpc/h, A {A:.3f}')

# check: the linear model against the mocks
mu, wmu = np.polynomial.legendre.leggauss(32)
def kaiser_fog(ell):
    L = np.polynomial.legendre.Legendre.basis(ell)(mu)
    K = (b1 + f * mu**2)**2 * np.exp(-(k[:, None] * mu * sigma_v)**2)
    return (2 * ell + 1) / 2 * np.sum(wmu * K * L, 1) * dilution * np.interp(k, k_lin, P_model)
fig, ax = plt.subplots(figsize=(WIDTH / 2, 2.4))
for ell, part in zip((0, 2), (P0, P2)):
    ax.plot(k, k * mean[part], color=MOCKS, ls=ELL_LINE[ell], label=rf'mocks $P_{ell}$')
    ax.plot(k, k * kaiser_fog(ell), color=SSC, ls=ELL_LINE[ell], label=rf'linear model $P_{ell}$')
ax.set(xlabel=K_LABEL, ylabel=r'$kP_\ell$', title='linear model used by the non-Gaussian terms')
ax.legend(fontsize=7)

## 5. Super-sample covariance

A density mode longer than the survey, $\delta_b$, shifts every $P_\ell(k)$ coherently. Two effects:

* **beat coupling**: the local power responds, $\Delta P_\ell(k)=d_k\sum_n R_\ell^{(n)}(k)\,D_n$, with responses $R$ from
  tree-level perturbation theory (growth, dilation, bias, redshift space; $n=0$ isotropic, $n=2$ tidal along the line of sight);
* **local average**: each mock normalises by its own data and randoms ($\alpha$, `norm`), which contain the same long mode:
  $-\hat P_\ell\,g_n(D^W_n+D^M_n)$, with $g_0=b_1+f/3$, $g_2=2f/3$.

So $C^{\rm SSC}=\sum_{XY}c^X(k)\,\sigma^2_{XY}\,c^Y(k')$: a handful of vectors $c^X$ (one per long-mode projection $X$)
times the long-mode variances $\sigma^2_{XY}$ of the window, a matrix of rank $\le4$.

In [ ]:
tracer_ssc = make_tracer(f'{TRACER}_{CAP}_ssc', 'mean', N_RANDOMS_LOCAL, shot_noise='catalogue')
cov_ssc = GaussianCovariance([tracer_ssc], spec.k_edges, ells=(0, 2, 4), L_max=4, n_sub=20000, n_near=300_000,
                             ds=2.0, ds_pair=10.0, s_split=80.0)       # used for its k bins, model and normalisation
cov_ssc.set_normalization(tracer_ssc.name, tracer_ssc.name, norm)
cov_ssc.set_model(model_from_mocks(tracer_ssc.name), masked=True)

def super_sample(local_average):
    ssc = SuperSampleCovariance(cov_ssc, (k_lin, P_model), b1=b1, f=f, b2=b2, bs2=bs2, local_average=local_average,
                                p_dressed=dressed, n_near=300_000, dilution=dilution, discreteness=True,
                                norm_kind='data-randoms')
    windows_file = f'{CACHE}/{TRACER}/ssc_windows_{TRACER}_{CAP}.npz'
    if os.path.exists(windows_file):
        ssc.load_windows(windows_file)
    ssc.compute_windows(tracer_ssc.name, verbose=True)             # only what the cache lacks
    os.makedirs(os.path.dirname(windows_file), exist_ok=True)
    ssc.save_windows(windows_file)
    return ssc

ssc = super_sample(local_average=True)
ssc_no_la = super_sample(local_average=False)
name = tracer_ssc.name

sigma2 = ssc.sigma2(name)                                  # long-mode variances of the window
for (x, y), v in sigma2.items():
    if x <= y:
        print(f'sigma^2 {x} x {y}: {v:.3e}')

In [ ]:
# the pieces: responses R_ell^(n)(k) and the coefficient vectors c^X (with and without the local average)
responses = ssc.responses()
coef, coef_no_la = ssc.coefficients(name, (0, 2, 4)), ssc_no_la.coefficients(name, (0, 2, 4))
fig, (left, right) = plt.subplots(1, 2, figsize=(WIDTH, 2.4))
Pk = np.interp(k, k_lin, P_model)
for (ell, n), R in responses.items():
    if np.max(np.abs(R)) > 1e-3 * np.max(Pk) and n <= 2 and ell <= 2:
        left.plot(k, R / Pk, ls=ELL_LINE[ell], color=SSC if n == 0 else T0, label=rf'$R_{ell}^{{({n})}}/P$')
left.axhline(0, color=MOCKS, lw=0.6)
left.set(xlabel=K_LABEL, title='responses to a long mode (units of P)')
left.legend(fontsize=7)
right.plot(k, coef_no_la[('W', 0)][P0] / mean[P0], color=SSC, ls=':', label='beat coupling only')
right.plot(k, coef[('W', 0)][P0] / mean[P0], color=SSC, label='+ local average (W)')
right.plot(k, coef[('M', 0)][P0] / mean[P0], color=SSC, ls='--', label='local average (M)')
right.axhline(0, color=MOCKS, lw=0.6)
right.set(xlabel=K_LABEL, title=r'$P_0$ response to $\delta_b$, units of $P_0$')
right.legend(fontsize=7)

In [ ]:
# the covariance is just c^T sigma^2 c (+ the Poisson part of the local average)
modes = list(coef)
c = np.stack([coef[x] for x in modes])
sigma2_matrix = np.array([[sigma2[(x, y)] for y in modes] for x in modes])
C_ssc_by_hand = c.T @ sigma2_matrix @ c + ssc.discreteness_covariance(name, (0, 2, 4))
C_ssc, _ = ssc.covariance([(name, name)])
C_ssc_no_la, _ = ssc_no_la.covariance([(name, name)])
print('by hand = thecov:', np.allclose(C_ssc_by_hand, C_ssc), '; rank of the long-mode part:',
      np.linalg.matrix_rank(c.T @ sigma2_matrix @ c, tol=1e-10 * np.abs(C_ssc).max()))

### 5b. A direct test: each mock's normalisation

The local average says that a mock's `norm` fluctuates with its long mode: $\delta_{\rm norm}=\mathcal N/\langle\mathcal N\rangle-1$.
thecov predicts its scatter and the regression slope of $\hat P_\ell(k)$ on $\delta_{\rm norm}$ (beat coupling minus local
average), for **fixed** weights and randoms. The mocks give both directly, with no fitting.

But the holi randoms copy each mock's redshifts and `NX`, so the FKP weights $1/(1+\bar nP_0)$ follow each mock's own
$n(z)$. A change $\delta$ of the shell density then changes $\bar n\,w_{\rm FKP}=\bar n/(1+\bar nP_0)$ only by
$\delta/(1+\bar nP_0)=\delta\,w_{\rm FKP}$: `norm` responds to the long mode with the factor
$c=\langle w_{\rm FKP}\rangle_{m^2}$ (~0.2 for LRG, ~0.85 for QSO). Measured from the spectra and the $n(z)$ of all mocks:
slopes of $\delta_{\rm norm}$ on $\delta\int n^2$ of 0.19 (LRG1) and 0.85 (QSO). $\hat P$ itself is divided by the same
realised weights, so this is mainly about what $\delta_{\rm norm}$ measures; `desi_validation/norm_fkp.py` tests the
mechanism mock by mock. The dashed lines rescale thecov's prediction by $c$ (scatter $\times c$, slope $/c$).

In [ ]:
pred = ssc.local_average_statistics(name, C_total=C_kernel + C_ssc)
fkp = np.asarray(cat.randoms['WEIGHT_FKP'], float)
m_own = alpha_all * rho_r * w_rand                          # ~m at every random (the response factor is a mean)
c_fkp = np.sum(w_rand * m_own * fkp) / np.sum(w_rand * m_own)    # <w_FKP> weighted by m^2 (alpha sum_r w_r m_r ... )
print(f'response factor of norm with per-mock FKP weights: c = <w_FKP>_m2 = {c_fkp:.3f}')
delta_norm = spec.norm / spec.norm.mean() - 1
slope_mocks = np.array([np.cov(spec.mocks[:, i], delta_norm)[0, 1] for i in range(3 * n_k)]) / delta_norm.var(ddof=1) / mean
slope_error = np.sqrt(np.diag(S)) / (np.sqrt(N) * delta_norm.std(ddof=1)) / np.abs(mean)
print(f'sigma(delta_norm): mocks {delta_norm.std(ddof=1):.4f}, thecov {pred["sigma_norm"]:.4f} '
      f'(clustering {pred["sigma_norm_clustering"]:.4f}, Poisson {pred["sigma_norm_poisson"]:.4f}); '
      f'with per-mock FKP weights {c_fkp * pred["sigma_norm"]:.4f}')
fig, axes = plt.subplots(1, 2, figsize=(WIDTH, 2.4))
for ax, ell, part in zip(axes, (0, 2), (P0, P2)):
    ax.errorbar(k, slope_mocks[part], slope_error[part], fmt='o', ms=2.5, lw=0.8, color=MOCKS, label='mocks')
    ax.plot(k, pred['slope'][part], color=SSC, label='thecov, fixed weights')
    ax.plot(k, pred['slope'][part] / c_fkp, color=SSC, ls='--', label='thecov, per-mock FKP weights')
    ax.axhline(0, color=MOCKS, lw=0.6)
    ax.set(xlabel=K_LABEL, title=rf'd ln $\hat P_{ell}$ / d $\delta_{{\rm norm}}$')
axes[0].legend()

## 6. Discreteness

Galaxies are discrete: four-point terms in which two or three of the four galaxies coincide add $B/\bar n$ (bispectrum,
one shared galaxy) and $P/\bar n^2$ (two shared). thecov computes them in the local approximation; then
`window_convolve` applies the window's mode mixing, read off the Gaussian covariance ($K=\mathrm{corr}(C_{\rm G})^{1/2}$).

In [ ]:
disc = DiscretenessCovariance(cov_ssc, (k_lin, P_model), b1=b1, f=f, b2=b2, bs2=bs2, sigma_fog=sigma_v,
                              **(dict(n_mu=16, n_phi=24) if sigma_v else {}))
parts = disc.components([(name, name)], ells=(0, 2, 4))      # {'B': ..., 'P': ...}, local approximation
C_disc_local = parts['B'] + parts['P']
C_disc = window_convolve(C_disc_local, C_kernel)

fig, ax = plt.subplots(figsize=(WIDTH / 2, 2.4))
for M, ls, label in ((parts['B'], '-', r'$B/\bar n$'), (parts['P'], '--', r'$P/\bar n^2$'), (C_disc, ':', 'total, window-convolved')):
    ax.plot(k, 100 * np.diag(M)[P0] / np.diag(C_kernel)[P0], color=DISC, ls=ls, label=label)
ax.set(xlabel=K_LABEL, ylabel='% of Gaussian variance', title=r'discreteness, $P_0$')
ax.legend()

## 7. (optional) The connected trispectrum

Tree-level $T_0$ and its response-based version. The mocks reject both (section 8f); kept for completeness.

In [ ]:
C_t0 = None
if RUN_T0:
    from thecov import TrispectrumCovariance, galileon_bias, response_split, response_covariance
    t0 = TrispectrumCovariance(cov_ssc, (k_lin, P_model), galileon_bias(b1, b2, bs2, b3=0.0), f=f,
                               n_workers=os.cpu_count(), sigma_fog=sigma_v, **(dict(n_mu=20, n_psi=40) if sigma_v else {}))
    tree = t0.components([(name, name)], ells=(0, 2, 4))
    collapsed = t0.collapsed([(name, name)], 0.06, ells=(0, 2, 4))
    split = response_split(window_convolve(tree['snake'] + tree['star'], C_kernel), k, 0.06,
                           C_kernel + C_ssc + C_disc, C_collapsed=window_convolve(collapsed, C_kernel))
    C_t0 = response_covariance(split)

## 8. Everything together, against the mocks

The recommended covariance, nothing fitted:
$$C = C^{\rm G}_{\xi\text{-kernel}} + C^{\rm SSC} + C^{\rm disc}.$$

In [ ]:
C_final = C_kernel + C_ssc + C_disc
models = [('own weight', C_own, OWN), ('local mean weight', C_mean, LOCAL), (r'$\xi$ kernel', C_kernel, GAUSS),
          ('+ SSC + discreteness', C_final, FINAL)]
for label, C, color in models:          # a covariance must be positive definite to be used
    lowest = np.linalg.eigvalsh(C / np.sqrt(np.outer(np.diag(C), np.diag(C))))[0]
    print(f'{label:22s} smallest eigenvalue of the correlation matrix: {lowest:.2e}' + ('' if lowest > 0 else '  NOT POSITIVE DEFINITE'))
np.savez(f'{RESULTS}/{TRACER}_{CAP}.npz', k=k, mocks=spec.mocks, mock_ids=spec.mock_ids, C_own=C_own, C_mean=C_mean,
         C_kernel=C_kernel, C_ssc=C_ssc, C_ssc_no_la=C_ssc_no_la, C_disc=C_disc, C_final=C_final,
         **({'C_t0': C_t0} if C_t0 is not None else {}))
print('saved', f'{RESULTS}/{TRACER}_{CAP}.npz')

### 8a. How big each term is (diagonal, % of the Gaussian variance)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(WIDTH, 2.4), sharey=True)
for ax, ell, part in zip(axes, (0, 2, 4), (P0, P2, P4)):
    gauss = np.diag(C_kernel)[part]
    ax.plot(k, 100 * np.diag(C_ssc)[part] / gauss, color=SSC, label='SSC')
    ax.plot(k, 100 * np.diag(C_ssc_no_la)[part] / gauss, color=SSC, ls=':', label='SSC, no local average')
    ax.plot(k, 100 * np.diag(C_disc)[part] / gauss, color=DISC, ls='--', label='discreteness')
    if C_t0 is not None:
        ax.plot(k, 100 * np.diag(C_t0)[part] / gauss, color=T0, ls='-.', label=r'$T_0$')
    ax.axhline(0, color=MOCKS, lw=0.6)
    ax.set(xlabel=K_LABEL, title=rf'$P_{ell}$')
axes[0].set_ylabel('% of Gaussian variance')
axes[0].legend(fontsize=7)

### 8b. Diagonals
The window fixes with the same SSC + discreteness added to each, then the recommended model against the Gaussian.

In [ ]:
extra = C_ssc + C_disc
plot_vs_mocks([('own weight', C_own + extra, OWN), ('local mean weight', C_mean + extra, LOCAL),
               (r'$\xi$ kernel', C_kernel + extra, GAUSS)], 'each window + SSC + discreteness')
plot_vs_mocks([(r'Gaussian ($\xi$ kernel)', C_kernel, GAUSS), ('+ SSC + discreteness', C_final, FINAL)], '')

### 8c. $\chi^2$
$\chi^2$ of each mock about the mean, and its average per degree of freedom using only $k<k_{\max}$
(1 if the covariance is right; grey: the expected scatter for $N$ mocks).

In [ ]:
from scipy.stats import chi2 as chi2_distribution
from paperstats import chi2_of_each_mock
n = 3 * n_k
fig, (left, right) = plt.subplots(1, 2, figsize=(WIDTH, 2.5))
chi2_values = [chi2_of_each_mock(spec.mocks, C) for label, C, color in models]
bins = np.linspace(min(n - 4 * np.sqrt(2 * n), np.percentile(chi2_values, 0.5)),
                   max(n + 4 * np.sqrt(2 * n), np.percentile(chi2_values, 99.5)), 45)
for values, (label, C, color) in zip(chi2_values, models):
    left.hist(values, bins=bins, histtype='step', color=color, lw=1.3, label=label)
left.plot(bins, chi2_distribution.pdf(bins, n) * N * (bins[1] - bins[0]), color=MOCKS, ls='--', lw=1, label=rf'$\chi^2_{{{n}}}$')
left.set(xlabel=r'$\chi^2$', ylabel='mocks'); left.legend(fontsize=6.5)
kmax_values = k[GROUP - 1::GROUP]
for label, C, color in models:
    values = []
    for kmax in kmax_values:
        sel = np.flatnonzero(np.tile(k <= kmax + 1e-9, 3))
        values.append(chi2_of_each_mock(spec.mocks[:, sel], C[np.ix_(sel, sel)]).mean() / len(sel))
    right.plot(kmax_values, values, color=color, label=label)
dof = np.array([3 * np.sum(k <= kmax + 1e-9) for kmax in kmax_values])
right.fill_between(kmax_values, 1 - np.sqrt(2 / (dof * N)), 1 + np.sqrt(2 / (dof * N)), color=LIGHT, lw=0)
right.axhline(1, color=MOCKS, lw=0.7)
right.set(xlabel=r'$k_{\max}$', ylabel=r'$\langle\chi^2\rangle$ / dof')

### 8d. The modes of the covariance

Whiten the mocks with a model: $y = C^{-1/2}(\hat P-\bar P)$. If $C$ is right, $y$ has unit variance in every
direction. The eigenvalues of $C^{-1/2}SC^{-1/2}$, divided by those of pure noise, show how many directions carry
missing variance. Then the mocks are split in two halves: the worst directions are found on half A and their
variance measured on half B (no bias from choosing them). Last: the shape of the worst direction of the Gaussian
model, compared with the leading direction of the SSC term.

In [ ]:
from paperstats import inverse_sqrt, null_eigenvalues, excess_directions, project
null = null_eigenvalues(N, n, n_draws=100)
null_median = np.median(null, axis=0)
low, high = np.percentile(null, [2.5, 97.5], axis=0) / null_median
rank = np.arange(1, n + 1)
fig, (left, mid, right) = plt.subplots(1, 3, figsize=(WIDTH, 2.5))
left.fill_between(rank, low, high, color=LIGHT, lw=0, label='noise (95%)')
for label, C, color in models:
    W = inverse_sqrt(C)
    left.plot(rank, np.sort(np.linalg.eigvalsh(W @ S @ W))[::-1] / null_median, color=color, label=label)
left.axhline(1, color=MOCKS, lw=0.7)
left.set(xscale='log', yscale='log', xlabel='rank', ylabel='eigenvalue / noise', title='whitened eigenvalues')
left.legend(fontsize=6)

order = np.random.default_rng(7).permutation(N)
half_A, half_B = spec.mocks[order[:N // 2]], spec.mocks[order[N // 2:]]
for (label, C, color), shift in zip(models[2:], (-0.15, 0.15)):
    _, directions, W = excess_directions(half_A, C)
    var_B = project(half_B, W, directions[:, :12]).var(0, ddof=1)
    mid.errorbar(np.arange(1, 13) + shift, var_B, var_B * np.sqrt(2 / (len(half_B) - 1)), fmt='o', ms=3, color=color, label=label)
mid.axhline(1, color=MOCKS, lw=0.7)
mid.set(xlabel='direction (found on half A)', ylabel='variance on half B', title='worst directions')
mid.legend(fontsize=6.5)

_, directions, W = excess_directions(spec.mocks, C_kernel)
worst = np.linalg.solve(W, directions[:, 0])                    # back to data space
lam, vec = np.linalg.eigh(W @ C_ssc @ W)
ssc_lead = np.linalg.solve(W, vec[:, -1])
for v, color, label in ((worst, MOCKS, 'mocks: worst direction'), (ssc_lead, SSC, 'SSC: leading direction')):
    v = v / np.sqrt(np.diag(C_kernel)); v = v * np.sign(v[np.argmax(np.abs(v))]) / np.max(np.abs(v))
    for ell, part in zip((0, 2), (P0, P2)):
        right.plot(k, v[part], color=color, ls=ELL_LINE[ell], label=label if ell == 0 else None)
right.axhline(0, color=MOCKS, lw=0.6)
right.set(xlabel=K_LABEL, title=r'shapes ($P_0$ solid, $P_2$ dashed), units of $\sigma_{\rm G}$')
right.legend(fontsize=6.5)

### 8e. Parameter errors
Each mock fitted with each covariance for an amplitude $A$, an extra quadrupole amplitude $A_2$, a dilation $\alpha$ and a
monopole constant $N$: scatter of the fits over the mocks / error predicted by the covariance (variance ratio).

In [ ]:
from paperstats import parameter_variance_ratio
from types import SimpleNamespace
sample = SimpleNamespace(k=k, mocks=spec.mocks, P0=lambda v: v[..., P0], P2=lambda v: v[..., P2], P4=lambda v: v[..., P4],
                         below=lambda kmax: np.flatnonzero(np.tile(k <= kmax + 1e-9, 3)))
kmax_values = np.arange(0.1, k[-1] + 1e-9, 0.025)
names = (r'amplitude $A$', r'quadrupole $A_2$', r'dilation $\alpha$', r'constant $N$')
fig, axes = plt.subplots(1, 4, figsize=(WIDTH, 2.3), sharey=True)
for label, C, color in models:
    ratios = np.array([parameter_variance_ratio(sample, C, kmax) for kmax in kmax_values])
    for j, ax in enumerate(axes):
        ax.plot(kmax_values, ratios[:, j], marker='o', ms=2.5, color=color, label=label)
for j, ax in enumerate(axes):
    ax.axhspan(1 - np.sqrt(2 / (N - 1)), 1 + np.sqrt(2 / (N - 1)), color=LIGHT, lw=0)
    ax.axhline(1, color=MOCKS, lw=0.7)
    ax.set(xlabel=r'$k_{\max}$', title=names[j])
axes[0].set_ylabel('variance ratio')
fig.legend(*axes[0].get_legend_handles_labels(), loc='lower center', ncol=4, bbox_to_anchor=(0.5, 1.0), fontsize=7)

### 8f. What amplitudes do the mocks prefer?
Fit $C = C^{\rm G} + a_{\rm SSC}C^{\rm SSC} + a_{\rm disc}C^{\rm disc}\,[+a_{T_0}C^{T_0}]$ to the mocks (Wishart likelihood).
The prediction is $a=1$.

In [ ]:
from paperstats import fit_amplitudes
templates = {'SSC': C_ssc, 'discreteness': C_disc}
if C_t0 is not None:
    templates['T0'] = C_t0
amplitude, error = fit_amplitudes(spec.mocks, C_kernel, templates)
for term in templates:
    print(f'{term:13s} a = {amplitude[term]:.2f} +- {error[term]:.2f}')